<!-- SPDX-License-Identifier: GPL-2.0-only -->
# 80 Parity

The notebook works through the page as a pupil of 2002 worked through 3D-Baukasten 1.3, and checks
that every step does what it did then: the mouse marks a body point by point in two views, the
fields turn from yellow to green, Einfügen inserts behind the selected line, a CSG group gathers the
bodies up to its CSG-Ende, `#Neu` and `Neu_Ende` make a new body, the tool bar and the tabs Ausgabe,
Kamera, Einstellung and Bilder do their work, and the colour and the texture reach the body.

In [ ]:
width = 1440
height = 900
look = 'modern'

In [ ]:
import sys, json, math
sys.path.insert(0, '..')
from harness import *

In [ ]:
site = Site()
page = Browser(site.url, width=width, height=height, theme=look)
page.open()
check(page.errors() == [], 'the page starts without an error', page.errors())

## 1. Datei › Neu

`mnuFileNew_Click` asks "Wollen Sie wirklich alle Objekte entfernen?" under the title FRAGE and
empties the scene on Ja.

In [ ]:
page.eval("() => { bausteine.act('neu'); }")
page.settle(300)
check(page.texts('#msgbox-title') == ['FRAGE'] and 'alle Objekte entfernen' in page.texts('#msgbox-text')[0], 'the question of 1.3')
page.page.click('#msgbox .c-default')
page.settle()
check(page.lines() == ['0 Raster'], 'Ja empties the scene', page.lines())

## 2. Marking a body with the mouse

A click on Quader in the body bar brings the tab Körper to the front, sets CB1 and starts the
handshake. A click in the Grundriss gives x and y, which turn green, and asks for z. A click in the
Aufriss gives z, frees the fields and asks for the size. Two more clicks give the size as the
difference to the point, and Einfügen inserts the body.

In [ ]:
page.page.click('#tb2 [data-body="Quader"]')
page.settle(200)
check(page.eval("() => document.querySelector('#cb1-edit').value") == 'Quader' and page.eval("() => document.querySelector('#f4').dataset.tab") == 'koerper', 'Quader in CB1, the tab Körper in front')
page.click_world(TOP, 2, 3, 0)
check(page.fields()[:3] == ['2*', '3*', '0'] and page.status() == 'z-Koordinate markieren', 'the Grundriss gives x and y and asks for z', page.fields(), page.status())
page.click_world(FRONT, 0, 3, 1)
check(page.fields()[:3] == ['2', '3', '1'] and page.status() == 'Grösse angeben oder markieren', 'the Aufriss gives z and asks for the size', page.fields(), page.status())
page.click_world(TOP, 6, 7, 0)
check(page.fields()[3:5] == ['4*', '4*'] and page.status() == 'Höhe markieren', 'the size in the Grundriss, then the height', page.fields(), page.status())
page.click_world(FRONT, 0, 7, 3)
check(page.fields()[3:6] == ['4', '4', '2'], 'the height in the Aufriss', page.fields())
page.page.click('#btn-einf')
page.settle()
items = page.eval("() => bausteine.app.scene.items.map((i) => [i.kind, i.position, i.size])")
check(items == [['CUBOID', {'x': 2, 'y': 3, 'z': 1}, {'x': 4, 'y': 4, 'z': 2}]], 'Einfügen inserts the marked Quader', items)
check(page.lines() == ['0 Raster', '1 Quader'] and page.status().startswith('Eingefügt'), 'the list and the status', page.lines())

## 3. A group of the CSG

CSG-Differenz opens a group, the next bodies go into it, CSG-Ende closes it, and the body after it
stands behind the group, as the flat list of 1.3 has it.

In [ ]:
page.entry('CSG-Differenz'); page.page.click('#btn-einf'); page.settle()
page.page.click('#tb2 [data-body="Würfel"]'); page.page.click('#btn-einf'); page.settle()
page.page.click('#tb2 [data-body="Zylinder"]'); page.page.click('#btn-einf'); page.settle()
page.entry('CSG-Ende'); page.page.click('#btn-einf'); page.settle(250)
page.page.click('#tb2 [data-body="Kugel"]'); page.page.click('#btn-einf'); page.settle()
check(page.lines() == ['0 Raster', '1 Quader', '2 CSG-Differenz', '3   -Würfel', '4   -Zylinder', '5 CSG-Ende', '6 Kugel'], 'the lines of 1.3', page.lines())

## 4. A new body

`#Neu` starts a new body, named Werkstück1 as in the thesis, `Neu_Ende` closes it, and its name
stands in CB1 from then on, from where it is placed again.

In [ ]:
page.entry('#Neu'); page.page.click('#btn-einf'); page.settle()
page.page.click('#tb2 [data-body="Kegel"]'); page.page.click('#btn-einf'); page.settle()
page.entry('Neu_Ende'); page.page.click('#btn-einf'); page.settle(250)
names = page.eval("() => [...document.querySelectorAll('#cb1 .c-combo-list [data-entry]')].map((e) => e.dataset.entry)")
check(names[-1] == 'Werkstück1', 'the new body is an entry of CB1', names[-3:])
page.entry('Werkstück1'); page.page.click('#btn-einf'); page.settle()
lines = page.lines()
check(lines[7:] == ['7 #Neu Werkstück1', '8 Kegel', '9 Werkstück1', '10 Werkstück1'], 'declared, ended and placed again', lines)
kinds = page.eval("() => bausteine.app.scene.items.map((i) => i.kind)")
check(kinds.count('DECLARE') == 1 and kinds.count('INSTANCE') == 1, 'one declaration and one instance', kinds)

## 5. Ändern and Löschen

A line of the list shows its body in the mask, Ändern applies the fields and keeps the selection,
Löschen removes the body.

In [ ]:
page.page.click('#cb2 [data-line="1"]'); page.settle(200)
check(page.fields()[:3] == ['2', '3', '1'], 'the Quader in the mask', page.fields())
page.page.fill('#t1-3', '2')
page.page.click('#btn-change'); page.settle()
quader = page.eval("() => bausteine.app.scene.items.find((i) => i.kind === 'CUBOID').position")
check(quader == {'x': 2, 'y': 3, 'z': 2} and page.eval("() => bausteine.app.line") == 1, 'Ändern moves it and keeps the selection', quader)
page.page.click('#btn-clear'); page.settle()
check(page.eval("() => bausteine.app.scene.items.some((i) => i.kind === 'CUBOID')") is False, 'Löschen removes it')
page.page.keyboard.press('Control+z'); page.settle()
check(page.eval("() => bausteine.app.scene.items.some((i) => i.kind === 'CUBOID')") is True, 'Strg+Z brings it back')

## 6. The tool bar

Position hides the body bar, Körper einfügen shows it again, 3D-Zoom shows the Schrägriss over all
four fields, Dia-Show brings the tab Bilder to the front, Raytrace starts the ray tracer of the
Schrägriss.

In [ ]:
page.page.click('#tb1 [data-tool="1"]'); page.settle(200)
check(not page.visible('#f2') and page.eval("() => bausteine.app.mode") == 'position', 'Position hides F2')
page.page.click('#tb1 [data-tool="2"]'); page.settle(200)
check(page.visible('#f2') and page.eval("() => bausteine.app.mode") == 'einfuegen', 'Körper einfügen shows F2')
page.page.click('#tb1 [data-tool="6"]'); page.settle(300)
check(page.visible('.view[data-view="3"]') and not page.visible('.view[data-view="1"]'), '3D-Zoom shows the Schrägriss over all four fields')
page.page.click('#tb1 [data-tool="6"]'); page.settle(300)
check(page.visible('.view[data-view="1"]'), 'a second click shows all four again')
page.page.click('#tb1 [data-tool="9"]'); page.settle(200)
check(page.eval("() => document.querySelector('#f4').dataset.tab") == 'bilder', 'Dia-Show brings the tab Bilder')
page.page.click('#tb1 [data-tool="5"]'); page.settle(400)
check(page.eval("() => bausteine.app.raytrace") is True and page.texts('#tb1 [data-tool="5"]', 'aria-pressed') == ['true'], 'Raytrace starts the ray tracer')
page.page.click('#tb1 [data-tool="5"]'); page.settle(200)

## 7. The 3D zoom and the 2D zoom

The buttons `<` zoom in and `>` zoom out, the 2D zoom on the ladder of 1.3 from 12 by 1.2, the 3D
zoom on its own ladder, and a click into the Schrägriss zooms it in without the normal views.

In [ ]:
page.eval("() => { bausteine.app.zoomf = 12; }")
page.page.click('#d2-zoomin'); page.settle(100)
check(abs(page.eval("() => bausteine.app.zoomf") - 14.4) < 1e-9, '< zooms 2D in by 1.2')
before = page.eval("() => [bausteine.app.zoomf, bausteine.app.d3zoomf]")
page.click_world(OBLIQUE, 0, 0, 0)
after = page.eval("() => [bausteine.app.zoomf, bausteine.app.d3zoomf]")
check(after[0] == before[0] and abs(after[1] - before[1] * 1.2) < 1e-9, 'a click in the Schrägriss zooms 3D alone', before, after)
page.page.click('#d3-zoomout'); page.settle(100)
check(abs(page.eval("() => bausteine.app.d3zoomf") - before[1]) < 1e-9, '> zooms 3D out')

## 8. Ausgabe

Übersetzen writes the text for POV-Ray into Text1 and enables Ausgeben. Ausgeben offers the file
and is disabled again.

In [ ]:
page.page.click('#sstab1 [data-tab="ausgabe"]'); page.settle(300)
page.page.click('#bpov-comp'); page.settle()
text = page.eval("() => document.querySelector('#text1').value")
check(text.startswith('// ====') and '#include "colors.inc"' in text and 'difference {' in text, 'Übersetzen writes the scene', text[:120])
check(page.eval("() => document.querySelector('#bpov-save').disabled") is False, 'Ausgeben is enabled')
with page.page.expect_download() as download:
    page.page.click('#bpov-save')
check(download.value.suggested_filename == 'ausgabe.pov', 'Ausgeben offers ausgabe.pov', download.value.suggested_filename)
check(page.eval("() => document.querySelector('#bpov-save').disabled") is True, 'and is disabled again')
check('pvengine.exe +W400 +H300' in page.texts('#pov-command')[0], 'the command line of 1.3 for POV-Ray')

## 9. Kamera

The sliders give the degrees still to turn about x, y and z, Start turns the Schrägriss by 2 degrees
every 100 ms, Einfügen takes its camera into the export.

In [ ]:
page.page.click('#sstab1 [data-tab="kamera"]'); page.settle(200)
page.page.fill('#hst-3', '20'); page.page.press('#hst-3', 'Enter')
page.page.click('#start'); page.settle(1500)
turn = page.eval("() => bausteine.app.turn")
check(abs(turn['z'] - 20) < 1e-9 and page.eval("() => document.querySelector('#hs-3').value") == '0', 'Start turns by the value of the slider', turn)
page.page.click('#btn1-1'); page.settle()
check('camera { location <15, 8, -15>' not in page.eval("() => document.querySelector('#text1').value"), 'Einfügen takes the camera of the Schrägriss')
page.page.click('#btn2-1'); page.settle()
check('location <15, 8, -15>' in page.eval("() => document.querySelector('#text1').value"), 'Löschen gives back the camera of 1.3')

## 10. Einstellung

The paths stay locked until the password 2002 of 1.3, Übernehmen keeps them and locks again.

In [ ]:
page.page.click('#sstab1 [data-tab="einstellung"]'); page.settle(200)
check(page.eval("() => document.querySelector('#tbau-home').disabled") is True, 'the paths are locked')
page.page.fill('#tpasswd', '2002')
check(page.eval("() => document.querySelector('#tbau-home').disabled") is False, '2002 unlocks them')
page.page.fill('#tbsp-def', 'haus')
page.page.click('#bdatei-change'); page.settle(200)
check(page.eval("() => document.querySelector('#tfname').value").endswith('haus') and page.eval("() => document.querySelector('#tbau-home').disabled") is True, 'Übernehmen keeps the name and locks again')
page.page.fill('#tpasswd', '2002'); page.page.click('#bdatei-def'); page.settle(200)

## 11. Colour and texture

The colour dialog of Windows has 48 basic and 16 custom colours, OK gives the colour to the selected
body, and the frame FARBE of F6 shows it. TEXTUR gives the body a texture of POV-Ray.

In [ ]:
page.page.click('#sstab1 [data-tab="koerper"]'); page.settle(200)
page.page.click('#cb2 [data-line="1"]'); page.settle(200)
page.page.click('#bcolor'); page.settle(200)
check(page.eval("() => document.querySelectorAll('.farben-grund .farben-swatch').length") == 48 and page.eval("() => document.querySelectorAll('.farben-custom .farben-swatch').length") == 16, '48 basic and 16 custom colours')
page.page.click('.farben-grund [data-color="#FF00FF"]')
page.page.click('#farben-ok'); page.settle()
quader = page.eval("() => bausteine.app.scene.items.find((i) => i.kind === 'CUBOID')")
check(quader['color'] == '#ff00ff', 'OK gives the colour to the body', quader['color'])
check(page.eval("() => getComputedStyle(document.querySelector('#frame-farbe')).backgroundColor") == 'rgb(255, 0, 255)', 'the frame FARBE shows it')
page.page.select_option('#cb3', 'Chrome_Metal'); page.settle()
check(page.eval("() => bausteine.app.scene.items.find((i) => i.kind === 'CUBOID').texture") == 'Chrome_Metal', 'TEXTUR gives the texture')

## 12. Bilder and the Dia-Show

Ansicht › Bild speichern keeps a picture of a view in the list of the tab Bilder, Anzeigen shows it
black over the screen, and End closes the show.

In [ ]:
with page.page.expect_download():
    page.eval("() => { bausteine.act('bild-speichern'); }")
page.settle(800)
page.page.click('#sstab1 [data-tab="bilder"]'); page.settle(400)
rows = page.eval("() => document.querySelectorAll('#file1 [data-index]').length")
check(rows >= 1, 'the picture is in the list', rows)
page.page.click('#bbild-anz'); page.settle(300)
check(page.visible('#f9') and page.eval("() => document.querySelector('#f9-picture').naturalWidth") > 0, 'Anzeigen shows the picture')
page.page.click('#bbild-ende'); page.settle(200)
check(not page.visible('#f9'), 'End closes the show')
check(page.errors() == [], 'no error on the way', page.errors())

In [ ]:
page.close()
site.stop()